In [1]:
import sys
sys.path.insert(0, "..") # Projektroot in Path
import os
os.chdir("..")

import yaml
from src.utils.seed import set_seed, get_device
from src.utils.data import make_food11_loaders
from src.models.vit import build_vit_from_cfg
from src.models.cnn_baseline import build_cnn_baseline
from src.utils.training import run_training
from src.utils.metrics import count_params, count_trainable_params
from src.nas.evolution import run_evolution

/Users/matsadel/Desktop/Master Informatik/MasterThesis/masterenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("configs/default.yaml") as f:
    cfg = yaml.safe_load(f)
    
set_seed(cfg["seed"])
device = get_device()
print("Device: ", device)

Device:  mps


In [3]:
train_loader, val_loader, test_loader, num_classes = make_food11_loaders(
    **cfg["dataset"]
    )

print(f"Classes: {num_classes} | Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}")

Classes: 11 | Train: 9866 | Val: 3430


In [4]:
model = build_vit_from_cfg(num_classes, cfg["model"])

print(f"Params: {count_params(model)/1e6:.1f}M | Trainable: {count_trainable_params(model)/1e6:.1f}M")

best_val, history = run_training(model, train_loader, val_loader, device, epochs=1)

Params: 21.7M | Trainable: 3.6M


KeyboardInterrupt: 

In [4]:
'''Cnn Baselines'''
resnet = build_cnn_baseline('resnet18.a1_in1k', num_classes)
effnet = build_cnn_baseline('efficientnet_b0.ra_in1k', num_classes)
mobilenet = build_cnn_baseline('mobilenetv3_large_100.ra_in1k', num_classes)

In [5]:
print(f"Params: {count_params(resnet)/1e6:.1f}M | Trainable: {count_trainable_params(resnet)/1e6:.1f}M")

best_val, history = run_training(resnet, train_loader, val_loader, device, epochs=1)

Params: 11.2M | Trainable: 11.2M
Epoch 001/1 | train_acc=0.6192 | val_acc=0.8064


In [6]:
print(f"Params: {count_params(mobilenet)/1e6:.1f}M | Trainable: {count_trainable_params(mobilenet)/1e6:.1f}M")

best_val, history = run_training(mobilenet, train_loader, val_loader, device, epochs=1)

Params: 4.2M | Trainable: 4.2M
Epoch 001/1 | train_acc=0.7556 | val_acc=0.8227


In [7]:
print(f"Params: {count_params(effnet)/1e6:.1f}M | Trainable: {count_trainable_params(effnet)/1e6:.1f}M")

best_val, history = run_training(effnet, train_loader, val_loader, device, epochs=1)

Params: 4.0M | Trainable: 4.0M
Epoch 001/1 | train_acc=0.7692 | val_acc=0.8638


# NAS Smoke test

In [4]:
from src.nas.evolution import run_evolution
from src.utils.data import make_food11_loaders
from src.utils.seed import get_device


In [5]:
device = get_device()
train_loader, val_loader, _, num_classes = make_food11_loaders(batch_size=32, num_workers=0)

In [6]:
results = run_evolution(
    num_classes=num_classes,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    pop_size=3,
    generations=2,
    proxy_epochs=2,
    log_path="results/nas_smoke_test.csv"
    )

print(f"\n{len(results)} Architekturen evaluiert")


=== Generation 0 | 3 Architekturen ===


Unexpected keys (blocks.10.attn.proj.bias, blocks.10.attn.proj.weight, blocks.10.attn.qkv.bias, blocks.10.attn.qkv.weight, blocks.10.mlp.fc1.bias, blocks.10.mlp.fc1.weight, blocks.10.mlp.fc2.bias, blocks.10.mlp.fc2.weight, blocks.10.norm1.bias, blocks.10.norm1.weight, blocks.10.norm2.bias, blocks.10.norm2.weight, blocks.11.attn.proj.bias, blocks.11.attn.proj.weight, blocks.11.attn.qkv.bias, blocks.11.attn.qkv.weight, blocks.11.mlp.fc1.bias, blocks.11.mlp.fc1.weight, blocks.11.mlp.fc2.bias, blocks.11.mlp.fc2.weight, blocks.11.norm1.bias, blocks.11.norm1.weight, blocks.11.norm2.bias, blocks.11.norm2.weight, blocks.4.attn.proj.bias, blocks.4.attn.proj.weight, blocks.4.attn.qkv.bias, blocks.4.attn.qkv.weight, blocks.4.mlp.fc1.bias, blocks.4.mlp.fc1.weight, blocks.4.mlp.fc2.bias, blocks.4.mlp.fc2.weight, blocks.4.norm1.bias, blocks.4.norm1.weight, blocks.4.norm2.bias, blocks.4.norm2.weight, blocks.5.attn.proj.bias, blocks.5.attn.proj.weight, blocks.5.attn.qkv.bias, blocks.5.attn.qkv.weight,

Epoch 001/2 | train_acc=0.1916 | val_acc=0.3050
Epoch 002/2 | train_acc=0.2366 | val_acc=0.4009
 val_acc=0.4009 | params=8.7M


Epoch 001/2 | train_acc=0.3202 | val_acc=0.3603
Epoch 002/2 | train_acc=0.3274 | val_acc=0.3332
 val_acc=0.3603 | params=21.7M
Epoch 001/2 | train_acc=0.2650 | val_acc=0.3758
Epoch 002/2 | train_acc=0.3094 | val_acc=0.4397
 val_acc=0.4397 | params=22.9M

=== Generation 1 | 3 Architekturen ===
Epoch 001/2 | train_acc=0.3049 | val_acc=0.3169
Epoch 002/2 | train_acc=0.3265 | val_acc=0.3230
 val_acc=0.3230 | params=21.7M
Epoch 001/2 | train_acc=0.2074 | val_acc=0.3446
Epoch 002/2 | train_acc=0.2186 | val_acc=0.3481
 val_acc=0.3481 | params=22.9M
Epoch 001/2 | train_acc=0.2946 | val_acc=0.3548
Epoch 002/2 | train_acc=0.3002 | val_acc=0.3956
 val_acc=0.3956 | params=22.9M

6 Architekturen evaluiert


In [9]:
import pandas as pd

df = pd.DataFrame([
    {k: v for k, v in r.items() if not k.startswith("_")}
    for r in results
])

df.sort_values("val_acc", ascending=False).style.format({
    "val_acc": "{:.4f}",
    "params": "{:.0f}",
    "lr": "{:.0e}",
})




,depth,num_heads,use_spt,use_lsa,finetune_mode,mixup_alpha,cutmix_prob,lr,weight_decay,val_acc,params,time_sec,generation
2,12,4,1,0,last4,0.400000,0.500000,1e-04,0.000100,0.4397,22850315,274.672714,0
0,4,4,1,0,full,0.000000,1.000000,1e-04,0.000100,0.4009,8654603,249.911391,0
5,12,6,1,0,last2,0.000000,0.500000,3e-04,0.000100,0.3956,22850315,252.393911,1
1,12,6,0,0,head,0.000000,0.000000,1e-03,0.000100,0.3603,21669899,205.114415,0
4,12,6,1,0,full,0.200000,0.500000,1e-04,0.000500,0.3481,22850315,416.171129,1
3,12,4,0,0,last4,0.000000,0.000000,1e-04,0.000500,0.3230,21669899,251.010580,1


# Proxy Test

In [10]:
from src.proxies.evaluate_proxy import evaluate_proxies
from src.nas.search_space import sample_config
from src.utils.seed import get_device

device = get_device()
train_loader, val_loader, _, num_classes = make_food11_loaders(
    batch_size=16, num_workers=0
)

# Einen Batch holen
data_batch, labels = next(iter(train_loader))

cfg = sample_config()
scores = evaluate_proxies(cfg, num_classes, data_batch, labels, device)
print(scores)

Unexpected keys (blocks.10.attn.proj.bias, blocks.10.attn.proj.weight, blocks.10.attn.qkv.bias, blocks.10.attn.qkv.weight, blocks.10.mlp.fc1.bias, blocks.10.mlp.fc1.weight, blocks.10.mlp.fc2.bias, blocks.10.mlp.fc2.weight, blocks.10.norm1.bias, blocks.10.norm1.weight, blocks.10.norm2.bias, blocks.10.norm2.weight, blocks.11.attn.proj.bias, blocks.11.attn.proj.weight, blocks.11.attn.qkv.bias, blocks.11.attn.qkv.weight, blocks.11.mlp.fc1.bias, blocks.11.mlp.fc1.weight, blocks.11.mlp.fc2.bias, blocks.11.mlp.fc2.weight, blocks.11.norm1.bias, blocks.11.norm1.weight, blocks.11.norm2.bias, blocks.11.norm2.weight, blocks.6.attn.proj.bias, blocks.6.attn.proj.weight, blocks.6.attn.qkv.bias, blocks.6.attn.qkv.weight, blocks.6.mlp.fc1.bias, blocks.6.mlp.fc1.weight, blocks.6.mlp.fc2.bias, blocks.6.mlp.fc2.weight, blocks.6.norm1.bias, blocks.6.norm1.weight, blocks.6.norm2.bias, blocks.6.norm2.weight, blocks.7.attn.proj.bias, blocks.7.attn.proj.weight, blocks.7.attn.qkv.bias, blocks.7.attn.qkv.weight,

{'params': 12203555, 'synflow': 58.42201613787353, 'naswot': 195.48599243164062, 'gradnorm': 32.73668976226881}


# Smoke Test with Proxies

In [5]:
results = run_evolution(
    num_classes=num_classes,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    pop_size=4,
    generations=2,
    proxy_epochs=2,
    log_path="results/nas_smoke_proxies.csv",
    use_proxies=True,
    proxy_keep_frac=0.5,
)


=== Generation 0 | 4 Architekturen ===


Unexpected keys (blocks.10.attn.proj.bias, blocks.10.attn.proj.weight, blocks.10.attn.qkv.bias, blocks.10.attn.qkv.weight, blocks.10.mlp.fc1.bias, blocks.10.mlp.fc1.weight, blocks.10.mlp.fc2.bias, blocks.10.mlp.fc2.weight, blocks.10.norm1.bias, blocks.10.norm1.weight, blocks.10.norm2.bias, blocks.10.norm2.weight, blocks.11.attn.proj.bias, blocks.11.attn.proj.weight, blocks.11.attn.qkv.bias, blocks.11.attn.qkv.weight, blocks.11.mlp.fc1.bias, blocks.11.mlp.fc1.weight, blocks.11.mlp.fc2.bias, blocks.11.mlp.fc2.weight, blocks.11.norm1.bias, blocks.11.norm1.weight, blocks.11.norm2.bias, blocks.11.norm2.weight, blocks.4.attn.proj.bias, blocks.4.attn.proj.weight, blocks.4.attn.qkv.bias, blocks.4.attn.qkv.weight, blocks.4.mlp.fc1.bias, blocks.4.mlp.fc1.weight, blocks.4.mlp.fc2.bias, blocks.4.mlp.fc2.weight, blocks.4.norm1.bias, blocks.4.norm1.weight, blocks.4.norm2.bias, blocks.4.norm2.weight, blocks.5.attn.proj.bias, blocks.5.attn.proj.weight, blocks.5.attn.qkv.bias, blocks.5.attn.qkv.weight,

  Proxies: 2/4 Kandidaten behalten
Epoch 001/2 | train_acc=0.1836 | val_acc=0.2574
Epoch 002/2 | train_acc=0.2185 | val_acc=0.3064
 val_acc=0.3064 | params=22.9M


Unexpected keys (blocks.10.attn.proj.bias, blocks.10.attn.proj.weight, blocks.10.attn.qkv.bias, blocks.10.attn.qkv.weight, blocks.10.mlp.fc1.bias, blocks.10.mlp.fc1.weight, blocks.10.mlp.fc2.bias, blocks.10.mlp.fc2.weight, blocks.10.norm1.bias, blocks.10.norm1.weight, blocks.10.norm2.bias, blocks.10.norm2.weight, blocks.11.attn.proj.bias, blocks.11.attn.proj.weight, blocks.11.attn.qkv.bias, blocks.11.attn.qkv.weight, blocks.11.mlp.fc1.bias, blocks.11.mlp.fc1.weight, blocks.11.mlp.fc2.bias, blocks.11.mlp.fc2.weight, blocks.11.norm1.bias, blocks.11.norm1.weight, blocks.11.norm2.bias, blocks.11.norm2.weight, blocks.4.attn.proj.bias, blocks.4.attn.proj.weight, blocks.4.attn.qkv.bias, blocks.4.attn.qkv.weight, blocks.4.mlp.fc1.bias, blocks.4.mlp.fc1.weight, blocks.4.mlp.fc2.bias, blocks.4.mlp.fc2.weight, blocks.4.norm1.bias, blocks.4.norm1.weight, blocks.4.norm2.bias, blocks.4.norm2.weight, blocks.5.attn.proj.bias, blocks.5.attn.proj.weight, blocks.5.attn.qkv.bias, blocks.5.attn.qkv.weight,

Epoch 001/2 | train_acc=0.2002 | val_acc=0.2767
Epoch 002/2 | train_acc=0.2663 | val_acc=0.3840
 val_acc=0.3840 | params=8.7M

=== Generation 1 | 4 Architekturen ===


Unexpected keys (blocks.10.attn.proj.bias, blocks.10.attn.proj.weight, blocks.10.attn.qkv.bias, blocks.10.attn.qkv.weight, blocks.10.mlp.fc1.bias, blocks.10.mlp.fc1.weight, blocks.10.mlp.fc2.bias, blocks.10.mlp.fc2.weight, blocks.10.norm1.bias, blocks.10.norm1.weight, blocks.10.norm2.bias, blocks.10.norm2.weight, blocks.11.attn.proj.bias, blocks.11.attn.proj.weight, blocks.11.attn.qkv.bias, blocks.11.attn.qkv.weight, blocks.11.mlp.fc1.bias, blocks.11.mlp.fc1.weight, blocks.11.mlp.fc2.bias, blocks.11.mlp.fc2.weight, blocks.11.norm1.bias, blocks.11.norm1.weight, blocks.11.norm2.bias, blocks.11.norm2.weight, blocks.4.attn.proj.bias, blocks.4.attn.proj.weight, blocks.4.attn.qkv.bias, blocks.4.attn.qkv.weight, blocks.4.mlp.fc1.bias, blocks.4.mlp.fc1.weight, blocks.4.mlp.fc2.bias, blocks.4.mlp.fc2.weight, blocks.4.norm1.bias, blocks.4.norm1.weight, blocks.4.norm2.bias, blocks.4.norm2.weight, blocks.5.attn.proj.bias, blocks.5.attn.proj.weight, blocks.5.attn.qkv.bias, blocks.5.attn.qkv.weight,

  Proxies: 2/4 Kandidaten behalten
Epoch 001/2 | train_acc=0.4113 | val_acc=0.8047
Epoch 002/2 | train_acc=0.4787 | val_acc=0.8362
 val_acc=0.8362 | params=21.7M
Epoch 001/2 | train_acc=0.4920 | val_acc=0.8353
Epoch 002/2 | train_acc=0.6832 | val_acc=0.9076
 val_acc=0.9076 | params=21.7M
